# Query Optimization Techniques: Filter Pushdown, Pruning, and Join Order

**Module**: 10_performance_and_optimization  
**Notebook**: 03_query_optimization.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Techniques to write optimal queries: filter pushdown, column pruning, join reordering, and eliminating redundant computations.
- **Why does this topic exist?**: Writing clean, Catalyst-friendly code prevents unnecessary compute and network overhead.
- **What problem does it solve?**: Queries that scan 50 columns when only 2 are used.
- **Where is it used in real-world Data Engineering?**: Optimizing warehouse queries.

---

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Query_Opt_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
# Optimal: Select needed columns and filter early
optimized = orders.filter("status = 'Completed'").select("order_id", "customer_id", "unit_price")
optimized.explain("formatted")

---

## Experiment: Column Pruning I/O Impact

- **1. What are we testing?**: Compare execution time when selecting 2 columns vs all 50 columns.
- **2. What do we expect?**: Column pruning reduces memory deserialization time.

In [ ]:
t0 = time.time(); orders.select("order_id").count(); t_prune = time.time() - t0
t0 = time.time(); orders.count(); t_full = time.time() - t0
print(f"Pruned scan: {t_prune:.3f}s, Full scan: {t_full:.3f}s")

**3. What happened?**: Pruned scan executed faster.  
**4. Why did it happen?**: Only required columns were deserialized.  
**5. Practical Engineering Lesson**: Always select only the specific columns required for downstream operations.

---

## Practical Exercises

### Exercise 1 (Beginner): Filter Before Join
**Problem Statement**: Filter table prior to join.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Drop Unused Columns
**Problem Statement**: Drop channel and discount.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Join Order Optimization
**Problem Statement**: How join order affects broadcast decisions.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Filter Before Join):
```python
print('Filter orders before joining with customers.')
```

#### Solution 2 (Drop Unused Columns):
```python
orders.drop('channel','discount').show(2)
```

#### Solution 3 (Join Order Optimization):
```python
print('Catalyst CBO can reorder joins if table statistics are analyzed.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Optimization Rules**: Select only needed columns; filter as early as possible; broadcast small tables.

### Essential Technical Terms
- `Column Pruning`: Skipping unused columns.
- `Predicate Pushdown`: Pushing filters to storage.

### Common Engineering Mistakes
- **SELECT * everywhere**: Forces full deserialization of every column.

### Interview Questions on This Topic
- **Q: Why is filtering before a join beneficial?**
  - *A*: It minimizes the number of rows participating in the join and shuffle phases.

### What to Study Next
Open 04_shuffle_optimization.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')